# Part 4: Multi-Agent Orchestration

In [1]:
# --- Reconnect to Foundry (run this first after opening a new notebook) ---
import subprocess, sys, json, shutil
from pathlib import Path

def run_cmd(args, check=True):
    exe = shutil.which(args[0])
    if exe: args = [exe] + args[1:]
    result = subprocess.run(args, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result

SUFFIX = run_cmd(["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"]).stdout.strip()[:6]
RESOURCE_GROUP = f"rg-foundry-workshop-{SUFFIX}"

outputs = json.loads(run_cmd([
    "az", "deployment", "group", "show", "-g", RESOURCE_GROUP, "-n", "main",
    "--query", "properties.outputs", "-o", "json"
]).stdout)

PROJECT_ENDPOINT = outputs["projectEndpoint"]["value"]
MODEL_DEPLOYMENT_NAME = outputs["modelDeploymentName"]["value"]
APP_INSIGHTS_CONN_STR = outputs["appInsightsConnectionString"]["value"]

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool, WebSearchPreviewTool, FunctionTool
from azure.identity import DefaultAzureCredential, AzureCliCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
print(f"✅ Connected to: {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

✅ Connected to: https://fndry-ws-c9db47.services.ai.azure.com/api/projects/workshop-project
   Model: gpt-4.1


---
## Section 5: Multi-Agent Orchestration (~15 min)

The **Agent Framework SDK** (`agent-framework`) provides orchestration patterns
for coordinating multiple agents. These agents run **in-memory** — ideal for
prototyping complex workflows.

| Pattern | Description | When to use |
|---------|-------------|-------------|
| **Sequential** | Agent A → Agent B → Agent C | Pipelines, content transformation |
| **Concurrent** | Fan-out / fan-in with aggregation | Parallel expert analysis |
| **Handoff** | Router delegates to specialists | Customer support, triage |
| **Group Chat** | Agents discuss collaboratively | Brainstorming, debate, review |

> Agent Framework automatically emits OpenTelemetry spans — visible in Foundry portal → Tracing.

In [2]:
# --- 5.0 Setup Agent Framework ---
import asyncio
from dataclasses import dataclass
from typing import Annotated
from agent_framework import (
    Agent,
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    AgentResponse,
    Case,
    Default,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)
from agent_framework.foundry import FoundryChatClient, FoundryAgent
from agent_framework.orchestrations import (
    HandoffBuilder,
    GroupChatBuilder,
)
from azure.ai.projects.models import MCPTool, WebSearchPreviewTool
from typing_extensions import Never
from agent_framework import InMemoryCheckpointStorage

# Create a shared chat client connected to the Foundry project
af_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT_NAME,
    credential=AzureCliCredential(),
)


print("✅ Agent Framework client ready")

✅ Agent Framework client ready


In [3]:
# --- Define Specialist Agents ---
# These are universally relatable personas — not tied to any specific industry.

researcher = Agent(
    client=af_client, name="Researcher",
    instructions=(
        "You are a thorough researcher. Gather key facts, data points, "
        "and insights on the given topic. Be specific and cite numbers when possible. "
        "Keep your research summary under 200 words."
    ),
)

writer = Agent(
    client=af_client, name="Writer",
    instructions=(
        "You are a professional writer. Transform research notes or raw input "
        "into polished, well-structured documents. Use clear headings, concise "
        "paragraphs, and professional tone. Keep output under 300 words."
    ),
)

critic = Agent(
    client=af_client, name="Critic",
    instructions=(
        "You are a constructive critic. Review the content for accuracy, "
        "completeness, clarity, and persuasiveness. Provide 3-5 specific, "
        "actionable improvement suggestions. Be direct but respectful."
    ),
)

customer_support = Agent(
    client=af_client, name="CustomerSupport",
    instructions=(
        "You handle customer service inquiries. Be empathetic, provide "
        "clear solutions, and confirm the customer's issue is resolved. "
        "If you can't resolve it, explain next steps."
    ),
)

travel_planner = Agent(
    client=af_client, name="TravelPlanner",
    instructions=(
        "You create detailed travel itineraries. Include flights, "
        "accommodation, key activities, estimated costs, and practical "
        "tips. Format as a day-by-day plan."
    ),
)

budget_analyst = Agent(
    client=af_client, name="BudgetAnalyst",
    instructions=(
        "You analyze budgets and costs. Break down expenses, identify "
        "savings opportunities, and provide financial recommendations. "
        "Always include a summary table."
    ),
)

print("✅ 6 specialist agents defined: Researcher, Writer, Critic, CustomerSupport, TravelPlanner, BudgetAnalyst")

✅ 6 specialist agents defined: Researcher, Writer, Critic, CustomerSupport, TravelPlanner, BudgetAnalyst


In [4]:
# --- 5.1 Sequential Orchestration ---
# Writer → Translator → Reviewer: write a tagline, translate to German, then review.
# chain_only_agent_responses=True: each agent sees only the previous agent's output.
# Uses WorkflowBuilder for graph visualization + Simple Text input in DevUI.

from agent_framework.orchestrations import SequentialBuilder
from agent_framework import AgentResponseUpdate

print("=" * 60)
print("SEQUENTIAL: Writer → Translator (DE) → Reviewer")
print("=" * 60)

seq_writer = Agent(
    client=af_client, name="Writer",
    instructions="You are a concise copywriter. Provide a single, punchy marketing sentence based on the prompt.",
)

seq_translator = Agent(
    client=af_client, name="Translator",
    instructions="You are a translator. Translate the given text into German. Output only the translation.",
)

seq_reviewer = Agent(
    client=af_client, name="Reviewer",
    instructions=(
        "You are a reviewer. Evaluate the quality of the German marketing tagline. "
        "Check grammar, tone, and impact. Provide a brief assessment (2-3 sentences)."
    ),
)

seq_workflow = (
    WorkflowBuilder(name="Sequential Writer-Translator-Reviewer", start_executor=seq_writer)
    .add_chain([seq_writer, seq_translator, seq_reviewer])
    .build()
)

# Run with streaming to see each agent's output as it arrives
last_agent = None
async for event in seq_workflow.run(
    "Write a tagline for a budget-friendly electric bike for urban commuters.",
    stream=True,
):
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        if event.data.author_name != last_agent:
            last_agent = event.data.author_name
            print(f"\n\n[{last_agent}]:", end=" ")
        print(event.data.text, end="", flush=True)

print("\n\n✅ Sequential orchestration complete (Writer → Translator DE → Reviewer)")


SEQUENTIAL: Writer → Translator (DE) → Reviewer


[Writer]: Ride smart, save big—your city commute just got electric.

[Translator]: Fahr clever, spare Geld – dein urbaner Weg wird elektrisch.

[Reviewer]: Bewertung:  
Der Slogan „Fahr clever, spare Geld – dein urbaner Weg wird elektrisch“ ist grammatikalisch korrekt und trifft einen freundlichen, klaren Ton. Die Botschaft ist prägnant und betont sowohl die Sparsamkeit als auch die Innovation, was besonders für preisbewusste urbane Pendler attraktiv wirkt. Insgesamt ein effektiver und einladender Marketing-Tagline.

✅ Sequential orchestration complete (Writer → Translator DE → Reviewer)


In [5]:
# --- 5.1b Sequential with FoundryAgent (server-side agents → traces in portal) ---
# Same Writer → Translator → Reviewer workflow, but using FoundryAgent backed by
# Prompt Agents registered in Foundry. Each agent invocation goes through the
# Foundry agent runtime, producing server-side traces visible in Portal → Tracing.

# Register the agents as Foundry Prompt Agents
writer_prompt = project_client.agents.create_version(
    agent_name="seq-writer",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions="You are a concise copywriter. Provide a single, punchy marketing sentence based on the prompt.",
    ),
)
print(f"✅ Foundry agent: {writer_prompt.name} v{writer_prompt.version}")

translator_prompt = project_client.agents.create_version(
    agent_name="seq-translator",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions="You are a translator. Translate the given text into German. Output only the translation.",
    ),
)
print(f"✅ Foundry agent: {translator_prompt.name} v{translator_prompt.version}")

reviewer_prompt = project_client.agents.create_version(
    agent_name="seq-reviewer",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a reviewer. Evaluate the quality of the German marketing tagline. "
            "Check grammar, tone, and impact. Provide a brief assessment (2-3 sentences)."
        ),
    ),
)
print(f"✅ Foundry agent: {reviewer_prompt.name} v{reviewer_prompt.version}")

# Wrap as FoundryAgent instances for the Agent Framework
foundry_writer = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT, agent_name=writer_prompt.name,
    agent_version=writer_prompt.version, credential=AzureCliCredential(), name="Writer",
)
foundry_translator = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT, agent_name=translator_prompt.name,
    agent_version=translator_prompt.version, credential=AzureCliCredential(), name="Translator",
)
foundry_reviewer = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT, agent_name=reviewer_prompt.name,
    agent_version=reviewer_prompt.version, credential=AzureCliCredential(), name="Reviewer",
)

# Build sequential workflow with FoundryAgent participants
print("\n" + "=" * 60)
print("SEQUENTIAL (FoundryAgent): Writer → Translator (DE) → Reviewer")
print("=" * 60)

seq_foundry = (
    WorkflowBuilder(name="Sequential FoundryAgent", start_executor=foundry_writer)
    .add_chain([foundry_writer, foundry_translator, foundry_reviewer])
    .build()
)

last_agent = None
async for event in seq_foundry.run(
    "Write a tagline for a budget-friendly electric bike for urban commuters.",
    stream=True,
):
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        if event.data.author_name != last_agent:
            last_agent = event.data.author_name
            print(f"\n\n[{last_agent}]:", end=" ")
        print(event.data.text, end="", flush=True)

print("\n\n✅ Sequential (FoundryAgent) complete!")
print("   Check Foundry Portal → Tracing — you should see traces for seq-writer, seq-translator, seq-reviewer.")


✅ Foundry agent: seq-writer v2
✅ Foundry agent: seq-translator v1
✅ Foundry agent: seq-reviewer v1

SEQUENTIAL (FoundryAgent): Writer → Translator (DE) → Reviewer


[Writer]: Ride smart, save more—your affordable urban e-bike awaits.

[Translator]: Fahre clever, spare mehr – dein günstiges E-Bike für die Stadt wartet auf dich.

[Reviewer]: Tagline:  
„Clever zur Arbeit, günstig unterwegs – dein E-Bike für die Stadt.“

Assessment:  
Die Grammatik ist korrekt und der Ton freundlich sowie motivierend. Die Kombination aus „clever“ und „günstig“ unterstreicht die Vorteile für preisbewusste, urbane Pendler und sorgt für eine positive, einladende Wirkung.

✅ Sequential (FoundryAgent) complete!
   Check Foundry Portal → Tracing — you should see traces for seq-writer, seq-translator, seq-reviewer.


In [6]:
# --- 5.2 Concurrent Fan-Out / Fan-In ---
# A dispatcher fans out the same prompt to 3 expert agents in parallel,
# then an aggregator joins their responses into a consolidated report.
# Uses custom Executor classes for proper dispatch and aggregation.

print("=" * 60)
print("CONCURRENT: Dispatcher → [Researcher + Marketer + Legal] → Aggregator")
print("=" * 60)

# --- Custom Executors ---

class DispatchToExperts(Executor):
    """Fans out the incoming prompt to all expert agents in parallel."""
    @handler
    async def dispatch(self, prompt: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        initial_message = Message("user", contents=[prompt])
        await ctx.send_message(AgentExecutorRequest(messages=[initial_message], should_respond=True))


@dataclass
class AggregatedInsights:
    research: str
    marketing: str
    legal: str


class AggregateInsights(Executor):
    """Joins expert responses into a single consolidated report."""
    @handler
    async def aggregate(self, results: list[AgentExecutorResponse], ctx: WorkflowContext[Never, str]) -> None:
        by_id: dict[str, str] = {}
        for r in results:
            by_id[r.executor_id] = r.agent_response.text
        consolidated = (
            f"📊 Consolidated Insights\n{'=' * 40}\n\n"
            f"🔍 Research Findings:\n{by_id.get('researcher', '')}\n\n"
            f"📣 Marketing Angle:\n{by_id.get('marketer', '')}\n\n"
            f"⚖️ Legal/Compliance Notes:\n{by_id.get('legal', '')}\n"
        )
        await ctx.yield_output(consolidated)


# --- Build the workflow ---

dispatcher = DispatchToExperts(id="dispatcher")
aggregator = AggregateInsights(id="aggregator")

expert_researcher = AgentExecutor(
    Agent(client=af_client, name="researcher",
          instructions="You're an expert market and product researcher. Given a prompt, provide concise, factual insights, opportunities, and risks.")
)
expert_marketer = AgentExecutor(
    Agent(client=af_client, name="marketer",
          instructions="You're a creative marketing strategist. Craft compelling value propositions and target messaging aligned to the prompt.")
)
expert_legal = AgentExecutor(
    Agent(client=af_client, name="legal",
          instructions="You're a cautious legal/compliance reviewer. Highlight constraints, disclaimers, and policy concerns based on the prompt.")
)

conc_workflow = (
    WorkflowBuilder(name="Fan-Out Fan-In Analysis", start_executor=dispatcher)
    .add_fan_out_edges(dispatcher, [expert_researcher, expert_marketer, expert_legal])
    .add_fan_in_edges([expert_researcher, expert_marketer, expert_legal], aggregator)
    .build()
)

# Run the workflow
async for event in conc_workflow.run(
    "Analyze the pros and cons of implementing a 4-day work week in a tech company.",
    stream=True,
):
    if event.type == "output" and event.executor_id == "aggregator":
        print(event.data)

print("\n✅ Concurrent fan-out/fan-in complete")


CONCURRENT: Dispatcher → [Researcher + Marketer + Legal] → Aggregator
📊 Consolidated Insights

🔍 Research Findings:
**Pros of Implementing a 4-Day Work Week in a Tech Company:**

1. **Increased Productivity:** Studies (Microsoft Japan, Perpetual Guardian NZ) show employees often accomplish the same or more in fewer hours due to improved focus and motivation.
2. **Talent Attraction & Retention:** A shorter workweek is a compelling differentiator in the competitive tech talent market, improving recruitment and reducing turnover.
3. **Employee Well-being:** More time off leads to reduced burnout, higher morale, and potentially fewer sick days.
4. **Cost Savings:** Potential reduction in operating costs (utilities, amenities) if offices are closed an extra day.
5. **Enhanced Employer Branding:** Being seen as progressive can boost reputation and media exposure.

**Cons of Implementing a 4-Day Work Week:**

1. **Collaboration Challenges:** Shortened weeks can hinder team alignment, especial

In [7]:
# --- 5.3 Handoff Orchestration ---
# A triage agent routes to the right specialist based on user intent.
# Specialists have function tools for their domain actions.
# HandoffBuilder auto-registers handoff tools for all participants.
# NOTE: Handoff uses Human-in-the-Loop (HIL) which is incompatible with DevUI.
#       We run a scripted demo here instead.

from agent_framework import tool
from agent_framework.orchestrations import HandoffAgentUserRequest

print("=" * 60)
print("HANDOFF: Triage → [Refund, Order, Return] Specialists")
print("=" * 60)

# --- Specialist function tools ---

@tool(approval_mode="never_require")
def process_refund(order_number: Annotated[str, "Order number to process refund for"]) -> str:
    """Simulated function to process a refund for a given order number."""
    return f"Refund processed successfully for order {order_number}."

@tool(approval_mode="never_require")
def check_order_status(order_number: Annotated[str, "Order number to check status for"]) -> str:
    """Simulated function to check the status of a given order number."""
    return f"Order {order_number} is currently being processed and will ship in 2 business days."

@tool(approval_mode="never_require")
def process_return(order_number: Annotated[str, "Order number to process return for"]) -> str:
    """Simulated function to process a return for a given order number."""
    return f"Return initiated successfully for order {order_number}. You will receive return instructions via email."

# --- Agents ---

triage_agent = Agent(
    client=af_client, name="triage_agent",
    instructions=(
        "You are frontline support triage. Route customer issues to the appropriate specialist agents "
        "based on the problem described."
    ),
    require_per_service_call_history_persistence=True,
)

refund_agent = Agent(
    client=af_client, name="refund_agent",
    instructions="You process refund requests.",
    tools=[process_refund],
    require_per_service_call_history_persistence=True,
)

order_agent = Agent(
    client=af_client, name="order_agent",
    instructions="You handle order and shipping inquiries.",
    tools=[check_order_status],
    require_per_service_call_history_persistence=True,
)

return_agent = Agent(
    client=af_client, name="return_agent",
    instructions="You manage product return requests.",
    tools=[process_return],
    require_per_service_call_history_persistence=True,
)

# --- Build handoff workflow ---
handoff_workflow = (
    HandoffBuilder(
        name="customer_support_handoff",
        participants=[triage_agent, refund_agent, order_agent, return_agent],
        checkpoint_storage=InMemoryCheckpointStorage(),
    )
    .with_start_agent(triage_agent)
    .build()
)
print("✅ Handoff workflow built: triage_agent → [refund_agent, order_agent, return_agent]")

# --- Run a scripted demo ---
scripted_responses = [
    "My order 1234 arrived damaged and the packaging was destroyed. I'd like to return it.",
    "Please also process a refund for order 1234.",
    "Thanks for resolving this.",
]

def print_events(events):
    """Process events and return pending HIL requests."""
    pending = []
    for event in events:
        if event.type == "handoff_sent":
            print(f"\n[🔀 Handoff: {event.data.source} → {event.data.target}]")
        elif event.type == "output":
            data = event.data
            if isinstance(data, AgentResponse):
                for m in data.messages:
                    if m.text:
                        print(f"- {m.author_name or m.role}: {m.text}")
        elif event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
            for m in event.data.agent_response.messages:
                if m.text:
                    print(f"- {m.author_name or m.role}: {m.text}")
            pending.append(event)
    return pending

print("\n[Starting handoff demo...]\n")
initial_message = "Hello, I need assistance with my recent purchase."
print(f"- User: {initial_message}")

workflow_result = handoff_workflow.run(initial_message, stream=True)
events = [event async for event in workflow_result]
pending_requests = print_events(events)

while pending_requests and scripted_responses:
    user_response = scripted_responses.pop(0)
    print(f"\n- User: {user_response}")
    responses = {
        req.request_id: HandoffAgentUserRequest.create_response(user_response)
        for req in pending_requests
    }
    new_events = await handoff_workflow.run(responses=responses)
    pending_requests = print_events(new_events)

print("\n✅ Handoff orchestration complete")


HANDOFF: Triage → [Refund, Order, Return] Specialists
✅ Handoff workflow built: triage_agent → [refund_agent, order_agent, return_agent]

[Starting handoff demo...]

- User: Hello, I need assistance with my recent purchase.
- triage_agent: I'd be happy to help! Could you please provide more details about the issue with your recent purchase? For example, are you looking to track your order, request a return or refund, or is there another problem? This will help me direct you to the right specialist.

- User: My order 1234 arrived damaged and the packaging was destroyed. I'd like to return it.

[🔀 Handoff: triage_agent → return_agent]
- return_agent: I'm sorry to hear your order arrived damaged. Your return request for order 1234 has been initiated. You will receive detailed return instructions via email shortly.

Is there anything else I can assist you with regarding your return, such as arranging a replacement or checking on a refund?
- return_agent: I'm sorry to hear your order arrive

In [8]:
# --- 5.4 Group Chat Orchestration ---
# An orchestrator coordinates multiple agents in a structured discussion.

print("=" * 60)
print("GROUP CHAT: Orchestrator + Researcher + Writer + Critic")
print("=" * 60)

orchestrator = Agent(
    client=af_client, name="Orchestrator",
    description="Coordinates multi-agent collaboration by selecting speakers",
    instructions=(
        "You coordinate a team discussion. Select the best next speaker based on "
        "what's needed: Researcher for facts, Writer for drafting, Critic for review. "
        "Aim for a productive 3-round discussion."
    ),
)

gc_workflow = (
    GroupChatBuilder(
        participants=[researcher, writer, critic],
        orchestrator_agent=orchestrator,
        termination_condition=lambda msgs: sum(1 for m in msgs if m.role == "assistant") >= 4,
        max_rounds=6,
    )
    .build()
)
gc_agent = gc_workflow.as_agent()

result = await gc_agent.run(
    "Create a compelling product launch announcement for a new AI-powered "
    "customer service platform that reduces response time by 80%."
)
if result.messages:
    for i, msg in enumerate(result.messages, 1):
        name = msg.author_name or msg.role
        if msg.text:
            print(f"\n[{i:02d} {name}]:")
            print(msg.text[:400])
            print("---")

print("\n✅ Group chat orchestration complete")


GROUP CHAT: Orchestrator + Researcher + Writer + Critic

[01 user]:
Create a compelling product launch announcement for a new AI-powered customer service platform that reduces response time by 80%.
---

[02 Researcher]:
Introducing **SwiftServe AI**, the revolutionary customer service platform designed to supercharge your support experience. Powered by advanced artificial intelligence, SwiftServe AI slashes response times by an incredible **80%**—ensuring your customers are never left waiting.

Gone are the days of long queues, automated replies, or frustrating delays. SwiftServe AI analyzes each inquiry in real 
---

[03 Critic]:
**Review of the Product Launch Announcement:**

**Accuracy:**  
The announcement accurately describes the core advantage (80% reduction in response time) and the main features of the platform. The claim is bold and clearly communicated.

**Completeness:**  
The content covers advantages, features, calls to action, and some integration options. It briefly ment

In [9]:
# --- 5.5 Custom Orchestration: RFX Multi-Agent Workflow ---
# A real-world pattern: answer RFP/RFI questions with quality assurance.
#
# Topology (visible as graph in DevUI):
#   input → QuestionAnswerer (MCP) → dispatcher → fan-out → [AnswerChecker, LinkChecker]
#                                                                    ↓              ↓
#                                                             fan-in → collector → manager
#                                                                                    ↓
#                                                           switch-case: APPROVE → output
#                                                                         reject → QA (loop)

print("=" * 60)
print("RFX Multi-Agent Workflow: QuestionAnswerer → [AnswerChecker + LinkChecker] → Manager")
print("=" * 60)

from link_checker import validate_urls_in_text

# --- Register Foundry Prompt Agents ---

# QuestionAnswerer with MCP (Microsoft Learn docs)
qa_prompt = project_client.agents.create_version(
    agent_name="rfx-question-answerer",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a question answerer for Microsoft technologies to help with RFP/RFQ/RFI situations. "
            "You take questions and provide detailed, accurate answers from the perspective of "
            "Microsoft Azure, AI, and cloud services.\n\n"
            "RULES:\n"
            "- ALWAYS use the MCP tool to search Microsoft Learn documentation before answering\n"
            "- Provide factual answers with concrete details (features, certifications, SLAs)\n"
            "- Include relevant Microsoft Learn documentation links from the MCP search results\n"
            "- If you don't know something, say so — don't fabricate information\n"
            "- Keep answers professional and suitable for enterprise procurement\n"
            "- If you receive feedback from checkers, fix ONLY those specific issues\n"
            "- Keep your answer under 400 words"
        ),
        tools=[MCPTool(
            server_label="microsoft-learn",
            server_url="https://learn.microsoft.com/api/mcp",
            require_approval="never",
        )],
    ),
)
print(f"✅ Foundry agent: {qa_prompt.name} v{qa_prompt.version}")

rfx_question_answerer = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT, agent_name=qa_prompt.name,
    agent_version=qa_prompt.version, credential=AzureCliCredential(), name="QuestionAnswerer",
)

# AnswerChecker with WebSearch
checker_prompt = project_client.agents.create_version(
    agent_name="rfx-answer-checker",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a QA checker for RFP/RFI responses about Microsoft technologies.\n\n"
            "VERIFICATION PROCESS:\n"
            "1. Use web search to independently verify key claims in the answer\n"
            "2. Check factual accuracy against your search results\n"
            "3. Check completeness — does the answer fully address the question?\n"
            "4. Check tone — professional and suitable for enterprise procurement?\n\n"
            "RESPONSE FORMAT:\n"
            "- If everything checks out, respond with ONLY: ANSWER CORRECT\n"
            "  Do NOT add verification notes or details when correct.\n"
            "- If issues found, start with 'ANSWER INCORRECT' then list each issue\n"
            "- Do NOT rewrite the answer — just identify the issues"
        ),
        tools=[WebSearchPreviewTool()],
    ),
)
print(f"✅ Foundry agent: {checker_prompt.name} v{checker_prompt.version}")

rfx_answer_checker = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT, agent_name=checker_prompt.name,
    agent_version=checker_prompt.version, credential=AzureCliCredential(), name="AnswerChecker",
)

# LinkChecker — in-memory Agent with URL validation function tool
rfx_link_checker = Agent(
    client=af_client, name="LinkChecker",
    description="Validates all URLs in the answer by making HTTP requests",
    instructions=(
        "You are a link checker. Your ONLY job is to validate URLs.\n\n"
        "PROCESS:\n"
        "1. Call validate_urls_in_text with the FULL text you received\n"
        "2. Return the function result EXACTLY as-is\n\n"
        "If the text has no URLs, respond with: LINKS CORRECT\n"
        "Do NOT make your own assessment — rely ONLY on the function results."
    ),
    tools=[validate_urls_in_text],
)

# --- Custom Executors for the workflow graph ---

class RFXInputReceiver(Executor):
    """Accepts plain text and converts to AgentExecutorRequest."""
    @handler
    async def receive(self, prompt: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        await ctx.send_message(AgentExecutorRequest(
            messages=[Message("user", contents=[prompt])], should_respond=True))

class RFXDispatcher(Executor):
    """Extracts QA answer text and fans out to checkers."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.last_answer = ""
    @handler
    async def dispatch(self, response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        self.last_answer = response.agent_response.text
        await ctx.send_message(AgentExecutorRequest(
            messages=[Message("user", contents=[self.last_answer])], should_respond=True))

@dataclass
class RFXCheckerVerdict:
    qa_answer: str
    answer_check: str
    link_check: str

class RFXCollector(Executor):
    """Fans in checker results into a single verdict."""
    def __init__(self, dispatcher_ref, **kwargs):
        super().__init__(**kwargs)
        self._dispatcher = dispatcher_ref
    @handler
    async def collect(self, results: list[AgentExecutorResponse], ctx: WorkflowContext[RFXCheckerVerdict]) -> None:
        by_id = {r.executor_id: r.agent_response.text for r in results}
        await ctx.send_message(RFXCheckerVerdict(
            qa_answer=self._dispatcher.last_answer,
            answer_check=by_id.get("AnswerChecker", ""),
            link_check=by_id.get("LinkChecker", ""),
        ))

@dataclass
class RFXManagerDecision:
    approved: bool
    content: str

class RFXManager(Executor):
    """Deterministic approve/reject — no LLM needed."""
    @handler
    async def decide(self, verdict: RFXCheckerVerdict, ctx: WorkflowContext[RFXManagerDecision]) -> None:
        answer_ok = verdict.answer_check.strip().upper().startswith("ANSWER CORRECT")
        links_ok = not verdict.link_check.strip().upper().startswith("LINKS INCORRECT")
        if answer_ok and links_ok:
            await ctx.send_message(RFXManagerDecision(approved=True, content=verdict.qa_answer))
        else:
            parts = []
            if not answer_ok:
                parts.append(f"AnswerChecker:\n{verdict.answer_check}")
            if not links_ok:
                parts.append(f"LinkChecker:\n{verdict.link_check}")
            await ctx.send_message(RFXManagerDecision(
                approved=False, content="REJECTED. Fix these issues:\n\n" + "\n\n".join(parts)))

class RFXFeedback(Executor):
    """Converts rejection feedback to AgentExecutorRequest for QA loop."""
    @handler
    async def send_feedback(self, decision: RFXManagerDecision, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        await ctx.send_message(AgentExecutorRequest(
            messages=[Message("user", contents=[decision.content])], should_respond=True))

class RFXApprovedOutput(Executor):
    """Emits the final approved answer."""
    @handler
    async def emit(self, decision: RFXManagerDecision, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output(decision.content)

# --- Build the workflow ---
rfx_input = RFXInputReceiver(id="rfx-input")
rfx_dispatcher = RFXDispatcher(id="rfx-dispatcher")
rfx_collector = RFXCollector(dispatcher_ref=rfx_dispatcher, id="rfx-collector")
rfx_manager = RFXManager(id="rfx-manager")
rfx_feedback = RFXFeedback(id="rfx-feedback")
rfx_output = RFXApprovedOutput(id="rfx-output")

rfx_workflow = (
    WorkflowBuilder(name="RFX Quality Assurance", start_executor=rfx_input, max_iterations=10)
    .add_edge(rfx_input, rfx_question_answerer)
    .add_edge(rfx_question_answerer, rfx_dispatcher)
    .add_fan_out_edges(rfx_dispatcher, [rfx_answer_checker, rfx_link_checker])
    .add_fan_in_edges([rfx_answer_checker, rfx_link_checker], rfx_collector)
    .add_edge(rfx_collector, rfx_manager)
    .add_switch_case_edge_group(rfx_manager, [
        Case(condition=lambda d: not d.approved, target=rfx_feedback),
        Default(target=rfx_output),
    ])
    .add_edge(rfx_feedback, rfx_question_answerer)
    .build()
)

# Run a test question
print("\n🔄 Running RFX QA workflow...")
async for event in rfx_workflow.run(
    "Does Azure support HIPAA compliance for healthcare workloads?",
    stream=True,
):
    if event.type == "output" and event.executor_id == "rfx-output":
        print(f"\n✅ APPROVED ANSWER:\n{event.data}")

print("\n✅ RFX QA workflow complete")


RFX Multi-Agent Workflow: QuestionAnswerer → [AnswerChecker + LinkChecker] → Manager
✅ Foundry agent: rfx-question-answerer v2
✅ Foundry agent: rfx-answer-checker v5

🔄 Running RFX QA workflow...

✅ APPROVED ANSWER:
Yes, Microsoft Azure supports HIPAA compliance for healthcare workloads.

Azure enables customers who are subject to HIPAA to use cloud services in a compliant manner by providing in-scope services, contractual commitments, and built-in technical safeguards. Azure offers a HIPAA Business Associate Agreement (BAA) as part of the Microsoft Product Terms, available to all covered entities or business associates under HIPAA. This BAA includes contractual assurances regarding data safeguarding, breach notifications, and access management in accordance with HIPAA and the HITECH Act.

Key points:
- Azure and Azure Government align with the NIST Cybersecurity Framework and are certified under ISO/IEC 27001. Controls align with other frameworks, such as CSA STAR and FedRAMP High.
- 

In [ ]:
# --- 5.6 DevUI: Interactive Agent & Workflow Dashboard ---
# Launch DevUI to interactively test all agents and workflows.
# All entities accept simple text input — just type your prompt!
# - Agents: chat interface
# - Workflows (WorkflowBuilder): graph visualization + Simple Text input
# - GroupChat: registered as agent (chat interface)

from agent_framework.devui import serve
import threading

# Collect all entities
entities = [
    # Individual agents — chat interface
    researcher,
    writer,
    critic,
    customer_support,
    travel_planner,
    budget_analyst,
    # Workflows — graph visualization + Simple Text input
    seq_workflow,       # Sequential: Writer → Translator (DE) → Reviewer
    conc_workflow,      # Fan-Out/Fan-In: Dispatcher → [Researcher + Marketer + Legal] → Aggregator
    rfx_workflow,       # RFX QA: QuestionAnswerer → [AnswerChecker, LinkChecker] → Manager (loop)
    # GroupChat — fresh instance for DevUI (the notebook cell's gc_workflow retains state)
    GroupChatBuilder(
        participants=[researcher, writer, critic],
        orchestrator_agent=orchestrator,
        termination_condition=lambda msgs: sum(1 for m in msgs if m.role == "assistant") >= 4,
        max_rounds=6,
    ).build().as_agent(name="Group Chat Discussion"),
]

print("🚀 Starting DevUI with all Section 5 agents and workflows...")
devui_thread = threading.Thread(
    target=serve,
    kwargs={
        "entities": entities,
        "port": 8080,
        "auto_open": False,
        "instrumentation_enabled": True,
    },
    daemon=True,
)
devui_thread.start()
import time; time.sleep(2)

print("✅ DevUI running at: http://localhost:8080")
print("\n   Available entities:")
print("   Agents (7): Researcher, Writer, Critic, CustomerSupport, TravelPlanner, BudgetAnalyst,")
print("               Group Chat Discussion")
print("   Workflows (3): Sequential Writer-Translator-Reviewer, Fan-Out Fan-In Analysis,")
print("                   RFX Quality Assurance")
print("\n   All entities accept simple text input — just type your prompt!")
print("   Workflows show the node graph with execution timeline.")
print("\n   NOTE: Handoff workflow uses HIL which is not yet supported in DevUI.")
print("         Test it by running the scripted demo in cell 9 (Section 5.3).")

INFO:     Started server process [22761]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8080 (Press CTRL+C to quit)


🚀 Starting DevUI with all Section 5 agents and workflows...
✅ DevUI running at: http://localhost:8080

   Available entities:
   Agents (7): Researcher, Writer, Critic, CustomerSupport, TravelPlanner, BudgetAnalyst,
               Group Chat Discussion
   Workflows (3): Sequential Writer-Translator-Reviewer, Fan-Out Fan-In Analysis,
                   RFX Quality Assurance

   All entities accept simple text input — just type your prompt!
   Workflows show the node graph with execution timeline.

   NOTE: Handoff workflow uses HIL which is not yet supported in DevUI.
         Test it by running the scripted demo in cell 9 (Section 5.3).


INFO:     127.0.0.1:62630 - "GET /meta HTTP/1.1" 200 OK
INFO:     127.0.0.1:62630 - "GET /v1/entities HTTP/1.1" 200 OK
INFO:     127.0.0.1:62630 - "GET /v1/conversations?agent_id=agent_in_memory_researcher_c6cccc333b0a4f469587a13f4b00c781 HTTP/1.1" 200 OK
INFO:     127.0.0.1:62630 - "POST /v1/conversations HTTP/1.1" 200 OK
INFO:     127.0.0.1:62635 - "GET /v1/conversations?agent_id=agent_in_memory_group-chat-discussion_40d16faca2a14b7ab26e654a3ff57f75 HTTP/1.1" 200 OK
INFO:     127.0.0.1:62635 - "POST /v1/conversations HTTP/1.1" 200 OK
INFO:     127.0.0.1:62651 - "POST /v1/responses HTTP/1.1" 200 OK


> 💡 **Check the traces**: Open **Foundry Portal → Tracing**. You'll see server-side traces
> for each agent invocation, showing tool calls, model inputs/outputs, and orchestration flow.

---


---
Proceed to `05-byod.ipynb`.